# Re-annoter le 800k (externe) et le 110k avec le prof bidoua-aware -- via rclone (sans drive.mount)

**Pourquoi rclone et pas `drive.mount()`** : le flux d'authentification
`dfs_ephemeral` de Colab echoue (400 `credentials-propagation`) sur ce
compte. On contourne en accedant a Drive via **rclone** avec le meme
token OAuth deja configure en local pour ce projet -- independant du flux
casse de Colab.

Principe (different de la version drive.mount) :
1. rclone est configure sur Colab a partir de TON token (colle via un
   champ masque, jamais ecrit en clair dans le notebook).
2. Les datasets d'entree sont **telecharges** depuis Drive vers le disque
   local Colab (rapide).
3. L'annotation tourne sur le disque local Colab. Le cache SQLite + les
   sorties sont **resynchronises periodiquement vers Drive** en tache de
   fond (pour survivre aux deconnexions, comme le faisait le mount).
4. En cas de deconnexion : re-executer les cellules dans l'ordre --
   l'etape 5 re-tire le cache depuis Drive, et l'annotation reprend
   exactement ou elle s'etait arretee (le cache deduplique le deja-fait).

**ATTENTION CACHE (piege reel)** : le cache d'annotations est indexe par
position + config du professeur, PAS par version du code d'evaluation.
Tous les dossiers de cache ci-dessous portent un suffixe `_bidoua_v1` :
ne jamais les faire pointer vers un ancien cache (pre-correctif), sinon
on recupere silencieusement les anciennes annotations sans bidoua.

**Debit a mesurer sur CE Colab avant le volume complet** (ne pas
supposer) : ~15-16s/position mesure autrefois sur l'externe, ~61s/position
mesure en local sur le 110k (budget de 60s presque toujours epuise). Deux
chiffres non interchangeables -> toujours piloter d'abord.


## 1 -- Configurer rclone : saisir le token dans le champ masque

**Recuperer le token** : sur ton Mac (avec le NOUVEAU token, apres
revocation + `rclone config reconnect songo_drive:`) :
```bash
python3 -c "import configparser,os; c=configparser.ConfigParser(); c.read(os.path.expanduser('~/.config/rclone/rclone.conf')); print(c['songo_drive']['token'])"
```
Ca affiche une ligne qui commence par `{"access_token":...`.

**IMPORTANT -- ne PAS coller le token dans le code de la cellule** (c'est
ce qui l'avait expose la derniere fois). Execute la cellule telle quelle :
un **champ de saisie masque apparait** (en haut de la fenetre dans VS
Code, ou juste sous la cellule sur le site Colab). Colle le token
DEDANS, puis Entree.

In [8]:
import os, subprocess, getpass

# Installer le binaire rclone sur Colab
subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True, check=False)

# >>> NE RIEN MODIFIER SUR CETTE LIGNE <<< : execute la cellule, puis colle
# le token dans le champ masque qui apparait (PAS dans le code ci-dessous).
token = getpass.getpass("Colle le token rclone ici puis Entree : ").strip()

assert token.startswith("{") and "access_token" in token, \
    "Token invalide : doit commencer par { et contenir access_token (colle-le dans le CHAMP, pas dans le code)"

os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[songo_drive]\ntype = drive\nscope = drive\ntoken = %s\n" % token)
os.chmod("/root/.config/rclone/rclone.conf", 0o600)
del token  # ne pas garder en memoire

# Test : doit lister le contenu de SongoFish/data sur ton Drive
print(subprocess.run("rclone lsd songo_drive:SongoFish/data", shell=True,
                     capture_output=True, text=True).stdout)

           0 2026-07-20 17:48:40        -1 .tmp.driveupload
           0 2026-07-20 12:53:34        -1 checkpoints
           0 2026-07-19 19:34:43        -1 dataset_curated_endgames
           0 2026-07-20 12:39:41        -1 dataset_curated_endgames (1)
           0 2026-07-20 12:54:04        -1 dataset_curated_endgames (1) (1)
           0 2026-07-20 12:54:05        -1 dataset_curated_endgames (1) (1) (1)
           0 2026-07-20 12:54:04        -1 dataset_full_matrix_merged_all_colabs
           0 2026-07-20 12:54:05        -1 dataset_v001_10k
           0 2026-07-16 17:39:25        -1 dataset_v002_100k
           0 2026-07-20 12:39:54        -1 dataset_v002_100k (1)
           0 2026-07-20 12:54:05        -1 dataset_v002_100k (1) (1)
           0 2026-07-20 12:56:51        -1 dataset_v002_100k (1) (1) (1)
           0 2026-07-16 21:01:37        -1 dataset_v003_110k
           0 2026-07-20 12:39:41        -1 dataset_v003_110k (1)
           0 2026-07-20 12:54:05        -1 dataset_v00

## 2 -- Cloner le depot (prive) et installer

Le depot `GlennEriss/songo-fish` est **prive** -> le clone exige un token
GitHub (un `git clone` anonyme echoue avec `could not read Username`).

**Creer un token GitHub (fine-grained, lecture seule sur ce repo)** :
1. GitHub -> Settings -> Developer settings -> **Personal access tokens**
   -> **Fine-grained tokens** -> *Generate new token*.
2. **Repository access** : *Only select repositories* -> `songo-fish`.
3. **Permissions** -> *Repository permissions* -> **Contents : Read-only**.
4. Generer, copier le token (commence par `github_pat_...`).

Execute la cellule ci-dessous : colle ce token dans le **champ masque**
qui apparait (PAS dans le code). Le token est retire de la config git
juste apres le clone, il ne reste pas stocke dans la session.

In [10]:
import getpass, subprocess

# >>> NE RIEN MODIFIER ICI <<< : execute la cellule, colle le token GitHub
# dans le champ masque qui apparait (PAS dans le code).
gh_token = getpass.getpass("Colle ton token GitHub (github_pat_...) puis Entree : ").strip()
assert gh_token, "token vide"

# Clone avec le token dans l'URL (construite en Python, non affichee).
# capture_output : ne jamais laisser fuiter l'URL-avec-token dans un traceback.
_url = f"https://{gh_token}@github.com/GlennEriss/songo-fish.git"
res = subprocess.run(["git", "clone", "-b", "dev", _url, "/content/songo-fish"],
                     capture_output=True, text=True)
del gh_token, _url  # ne pas garder le token en memoire
if res.returncode != 0:
    raise SystemExit("Echec du clone -- verifie que le token a bien l'acces Contents:Read sur songo-fish.")

# Retirer toute trace du token : remettre l'URL du remote sans identifiant.
subprocess.run(["git", "-C", "/content/songo-fish", "remote", "set-url", "origin",
                "https://github.com/GlennEriss/songo-fish.git"], check=True)
print("Clone OK (branche dev).")


Clone OK (branche dev).


In [11]:
%cd /content/songo-fish
!pip install -q -e ".[train]"


/content/songo-fish
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for songo-ai (pyproject.toml) ... done


In [12]:
import multiprocessing
NUM_WORKERS = multiprocessing.cpu_count()
print('coeurs disponibles sur cette session Colab:', NUM_WORKERS)


## 3 -- Chemins + telechargement des entrees + fonctions de synchro

Definit les chemins (Drive distant + miroir local Colab), telecharge les
datasets d'entree, re-tire les caches deja constitues (reprise), et
definit `start_bg_sync()` / `stop_bg_sync()` pour la resync automatique
vers Drive pendant les longues annotations.

In [13]:
import subprocess, os, signal

# --- Distant (Drive, via rclone) ---
REMOTE = "songo_drive:SongoFish/data"
R_EXTERNAL_IN  = f"{REMOTE}/dataset_full_matrix_merged_all_colabs"
R_EXTERNAL_OUT = f"{REMOTE}/dataset_external_reannotated_bidoua_v1"
R_OWN_IN       = f"{REMOTE}/dataset_v003_110k"
R_OWN_OUT      = f"{REMOTE}/dataset_v003_110k_bidoua_v1"

# --- Local (disque Colab, ephemere) ---
L = "/content/songo_data"
L_EXTERNAL_IN  = f"{L}/dataset_full_matrix_merged_all_colabs"
L_EXTERNAL_OUT = f"{L}/dataset_external_reannotated_bidoua_v1"
L_EXTERNAL_CACHE = f"{L_EXTERNAL_OUT}/annotation_cache"
L_OWN_IN       = f"{L}/dataset_v003_110k"
L_OWN_OUT      = f"{L}/dataset_v003_110k_bidoua_v1"
L_OWN_CACHE    = f"{L_OWN_OUT}/annotation_cache"

for d in (L_EXTERNAL_OUT, L_EXTERNAL_CACHE, L_OWN_OUT, L_OWN_CACHE):
    os.makedirs(d, exist_ok=True)

def rc(cmd):
    print(">", cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

# Telecharger les entrees Drive -> local (rapide, une fois)
rc(f'rclone copy "{R_EXTERNAL_IN}" "{L_EXTERNAL_IN}" --progress')
rc(f'rclone copy "{R_OWN_IN}" "{L_OWN_IN}" --progress')

# Reprise : re-tirer les caches + sorties deja constitues sur Drive (no-op au 1er lancement)
rc(f'rclone copy "{R_EXTERNAL_OUT}" "{L_EXTERNAL_OUT}"')
rc(f'rclone copy "{R_OWN_OUT}" "{L_OWN_OUT}"')

# --- Synchro periodique local -> Drive, en tache de fond ---
_sync_proc = None
def start_bg_sync(local_dir, remote_dir, every_s=300):
    global _sync_proc
    stop_bg_sync()
    script = (f'while true; do sleep {every_s}; '
              f'rclone copy "{local_dir}" "{remote_dir}" >/dev/null 2>&1; done')
    _sync_proc = subprocess.Popen(["bash", "-c", script], preexec_fn=os.setsid)
    print(f"[sync] tache de fond demarree : {local_dir} -> {remote_dir} toutes les {every_s}s")

def stop_bg_sync():
    global _sync_proc
    if _sync_proc is not None:
        os.killpg(os.getpgid(_sync_proc.pid), signal.SIGTERM)
        _sync_proc = None

def push(local_dir, remote_dir):
    rc(f'rclone copy "{local_dir}" "{remote_dir}"')
    print("[sync] push final termine")


> rclone copy "songo_drive:SongoFish/data/dataset_full_matrix_merged_all_colabs" "/content/songo_data/dataset_full_matrix_merged_all_colabs" --progress
Transferred:   	          0 B / 9.804 MiB, 0%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 5
Transferred:            0 / 5, 0%
Elapsed time:         0.5s
Transferring:
 *                                         Icon␍: transferring
 *                         dataset_metadata.json: transferring
 *                                      test.npz: transferring
 *                                     train.npz: transferringTransferred:   	    4.691 KiB / 4.691 KiB, 100%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 5
Transferred:            2 / 5, 40%
Elapsed time:         1.0s
Transferring:
 *                                      test.npz: transferring
 *                                     train.npz: transferring
 *                                validation.npz: transferringTransferred:   	    4.691 KiB / 4.691 KiB, 100%, 4

## Section A -- dataset externe (883k, vraies parties)

### A.1 -- pilote (mesurer le debit reel sur CE Colab)

In [ ]:
!python3 apps/trainer/scripts/reannotate_external_dataset.py \
    --input-dir "{L_EXTERNAL_IN}" \
    --out-dir /content/pilot_external_bidoua \
    --cache-dir "{L_EXTERNAL_CACHE}" \
    --limit 100 --num-workers {NUM_WORKERS}


### A.2 -- volume complet (une fois le debit du pilote juge acceptable)

La resync vers Drive tourne en fond (toutes les 5 min) + un push final.
Le cache etant sauvegarde, une deconnexion se reprend en re-executant les
cellules dans l'ordre (l'etape 3 re-tire le cache, l'annotation saute le
deja-fait).

In [ ]:
start_bg_sync(L_EXTERNAL_OUT, R_EXTERNAL_OUT, every_s=300)
try:
    !python3 apps/trainer/scripts/reannotate_external_dataset.py \
        --input-dir "{L_EXTERNAL_IN}" \
        --out-dir "{L_EXTERNAL_OUT}" \
        --cache-dir "{L_EXTERNAL_CACHE}" \
        --num-workers {NUM_WORKERS}
finally:
    stop_bg_sync()
    push(L_EXTERNAL_OUT, R_EXTERNAL_OUT)


## Section B -- notre dataset (110k, deja au format interne)

### B.1 -- pilote

In [ ]:
!python3 apps/trainer/scripts/reannotate_own_dataset.py \
    --input-dir "{L_OWN_IN}" \
    --out-dir /content/pilot_110k_bidoua \
    --cache-dir "{L_OWN_CACHE}" \
    --limit 30 --num-workers {NUM_WORKERS}


### B.2 -- volume complet

In [ ]:
start_bg_sync(L_OWN_OUT, R_OWN_OUT, every_s=300)
try:
    !python3 apps/trainer/scripts/reannotate_own_dataset.py \
        --input-dir "{L_OWN_IN}" \
        --out-dir "{L_OWN_OUT}" \
        --cache-dir "{L_OWN_CACHE}" \
        --num-workers {NUM_WORKERS}
finally:
    stop_bg_sync()
    push(L_OWN_OUT, R_OWN_OUT)


## Etape suivante (pas encore decidee) -- fusion

Une fois les DEUX re-annotations terminees et poussees sur Drive
(`R_EXTERNAL_OUT` et `R_OWN_OUT`), reste a decider comment les combiner
avec le reste (`dataset_v004_290k`, `dataset_v005_400k`, pas encore
re-annotes) -- volontairement pas fait automatiquement ici, c'est une
decision a prendre une fois les deux volumes reellement re-annotes et
leurs metriques regardees (pas un merge aveugle).